In [ ]:
# =====================================================================
# Langkah 0 & 1: Load Dataset & EDA Singkat
# =====================================================================
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("--- LANGKAH 1: LOAD & EDA SINGKAT ---")

df = sns.load_dataset('titanic')
# Pilih kolom yang akan digunakan
cols = ['pclass','sex','age','sibsp','parch','fare','embarked','survived']
df = df[cols].copy()
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())

print('\nDistribusi target:')
print(df['survived'].value_counts(normalize=True).round(3))
# survived=0: ~61.6%, survived=1: ~38.4% — kelas tidak seimbang!


--- LANGKAH 1: LOAD & EDA SINGKAT ---
Shape: (891, 8)

Missing values:
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
survived      0
dtype: int64

Distribusi target:
survived
0    0.616
1    0.384
Name: proportion, dtype: float64

# =====================================================================
# Langkah 2: Handling Missing Values
# =====================================================================

print("\n--- LANGKAH 2: HANDLING MISSING VALUES ---")

# Age: isi dengan median (robust terhadap outlier)
df['age'] = df['age'].fillna(df['age'].median())

# Embarked: isi dengan modus (nilai paling sering)
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

print('Missing setelah handling:')
print(df.isnull().sum()) # Semua harus 0

--- LANGKAH 2: HANDLING MISSING VALUES ---
Missing setelah handling:
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
survived    0
dtype: int64

# =====================================================================
# Langkah 3: Encoding Kategorikal
# =====================================================================
print("\n--- LANGKAH 3: ENCODING KATEGORIKAL ---")

# One-Hot Encoding untuk 'sex' dan 'embarked'
df = pd.get_dummies(df,
columns=['sex', 'embarked'],
drop_first=True, # hindari dummy variable trap
dtype=int)
print('Kolom setelah encoding:')
print(df.columns.tolist())
# ['pclass','age','sibsp','parch','fare','survived',
# 'sex_male','embarked_Q','embarked_S']

--- LANGKAH 3: ENCODING KATEGORIKAL ---
Kolom setelah encoding:
['pclass', 'age', 'sibsp', 'parch', 'fare', 'survived', 'sex_male', 'embarked_Q', 'embarked_S']

# =====================================================================
# Langkah 4: Train-Test Split
# =====================================================================

print("\n--- LANGKAH 4: TRAIN-TEST SPLIT ---")

# Memisahkan Fitur (X) dan Target (y)
X = df.drop('survived', axis=1)
y = df['survived']

# Split data 80% Train, 20% Test dengan Stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y # Menjaga proporsi kelas target tetap seimbang di train & test
)

print(f'Jumlah data Train: {X_train.shape[0]} baris')
print(f'Jumlah data Test : {X_test.shape[0]} baris')
print('\nProporsi kelas "survived" di Train:')
print(y_train.value_counts(normalize=True).round(3))
print('\nProporsi kelas "survived" di Test:')
print(y_test.value_counts(normalize=True).round(3))
print("-" * 50)

--- LANGKAH 4: TRAIN-TEST SPLIT ---
Jumlah data Train: 712 baris
Jumlah data Test : 179 baris

Proporsi kelas "survived" di Train:
survived
0    0.617
1    0.383
Name: proportion, dtype: float64

Proporsi kelas "survived" di Test:
survived
0    0.615
1    0.385
Name: proportion, dtype: float64
--------------------------------------------------

# =====================================================================
# Langkah 5: Feature Scaling
# =====================================================================
print("\n--- LANGKAH 5: FEATURE SCALING ---")
# Hanya kolom numerik kontinu/ordinal yang perlu di-scale
num_cols = ['pclass', 'age', 'sibsp', 'parch', 'fare']
scaler = StandardScaler()

# Penting: fit_transform HANYA di X_train
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# Penting: transform SAJA di X_test (menggunakan mean dan std dari X_train)
X_test[num_cols] = scaler.transform(X_test[num_cols])

print('Mean scaler (dari training set):', scaler.mean_.round(2))
print('Std scaler (dari training set) :', scaler.scale_.round(2))
print('\nContoh isi X_train setelah scaling (3 baris pertama):')
print(X_train.head(3).round(3))

print('\n' + "="*40)
print("DATA READY UNTUK PROSES MACHINE LEARNING!")
print(f'Final Shape X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'Final Shape X_test : {X_test.shape}, y_test : {y_test.shape}')
print("="*40)

--- LANGKAH 5: FEATURE SCALING ---
Mean scaler (dari training set): [ 2.31 29.46  0.49  0.39 31.82]
Std scaler (dari training set) : [ 0.83 13.03  1.06  0.84 48.03]

Contoh isi X_train setelah scaling (3 baris pertama):
     pclass    age  sibsp  parch   fare  sex_male  embarked_Q  embarked_S
692   0.830 -0.112 -0.465 -0.466  0.514         1           0           1
481  -0.371 -0.112 -0.465 -0.466 -0.663         1           0           1
527  -1.571 -0.112 -0.465 -0.466  3.955         1           0           1

========================================
DATA READY UNTUK PROSES MACHINE LEARNING!
Final Shape X_train: (712, 8), y_train: (712,)
Final Shape X_test : (179, 8), y_test : (179,)
========================================